# NexGen — Face Enrollment & Matching Prototype

Prototype for the enrollment/embedding module: register known people from a
few photos each, then check whether a new face matches someone registered
or should be flagged as unknown.

**Before running the enrollment cells below**, drop your own photos into:

```
data/enrollment/
    <person_name>/
        photo1.jpg
        photo2.jpg
        photo3.jpg
    <another_person>/
        ...
```

3-5 photos per person, varied angle/lighting, works best (that's what you
said you wanted for the demo). One clearly-visible face per photo — extra
people in frame will make enrollment for that photo fail loudly rather than
silently enrolling the wrong face.

In [2]:
import sys
from pathlib import Path

def find_project_root(marker: str = "requirements.txt") -> Path:
    """Walk up from the kernel's cwd until we find the project root.

    Needed because Path.cwd() is the *kernel's* working directory, not this
    notebook's file location -- in VS Code / some Jupyter setups that's the
    workspace root, not the notebooks/ folder, so a fixed "../" was fragile.
    """
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find project root (looked for {marker!r} above {here})."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

from enrollment import FaceEmbedder, IdentityStore, enroll_from_directory

DATA_DIR = PROJECT_ROOT / "data" / "enrollment"
STORE_PATH = PROJECT_ROOT / "data" / "identity_store.pkl"

embedder = FaceEmbedder()
print(f"Embedder ready on device: {embedder.device}")

Project root: /home/sohail/face-embedder


ModuleNotFoundError: No module named 'enrollment'

## Enroll everyone found under `data/enrollment/`

In [ ]:
store = IdentityStore()

people = [p for p in DATA_DIR.iterdir() if p.is_dir()] if DATA_DIR.exists() else []

if not people:
    print(
        f"No enrollment photos found under {DATA_DIR.resolve()}.\n"
        "Create a subfolder per person (e.g. data/enrollment/alice/) with a "
        "few photos each, then re-run this cell."
    )
else:
    reports = enroll_from_directory(DATA_DIR, embedder, store)
    for r in reports:
        print(
            f"{r['name']:15s} | {r['num_embeddings']} photo(s) enrolled "
            f"| min self-similarity {r['min_pairwise_similarity']:.2f}"
        )
    store.save(STORE_PATH)
    print(f"\nSaved {len(store.list_identities())} identities to {STORE_PATH}")

## Test matching

Point `TEST_IMAGE` at a photo to check against the enrolled identities —
ideally a *different* photo of an enrolled person (to sanity-check a true
match) and separately a photo of someone *not* enrolled (to sanity-check
`unknown`). This is also where you'd tune `threshold`: lower it if real
matches are being called unknown, raise it if strangers are matching.

In [ ]:
TEST_IMAGE = Path("../data/test_query.jpg")  # <- point this at a test photo
THRESHOLD = 0.65

if not TEST_IMAGE.exists():
    print(f"Set TEST_IMAGE to a real photo path (looked for {TEST_IMAGE.resolve()}).")
elif not store.list_identities():
    print("No identities enrolled yet — run the enrollment cell above first.")
else:
    faces = embedder.detect_and_embed(TEST_IMAGE)
    print(f"Detected {len(faces)} face(s) in {TEST_IMAGE.name}\n")
    for box, embedding, det_conf in faces:
        name, score = store.match(embedding, threshold=THRESHOLD)
        print(
            f"  box={[round(c) for c in box]} "
            f"detection_conf={det_conf:.2f} -> {name} (similarity={score:.3f})"
        )

## Next steps

- This uses a local pickle-backed `IdentityStore`. Same interface will swap
  onto Alibaba Tair (TairVector) or AnalyticDB vector search for the
  deployed version — `add_identity`/`match` are the two methods to
  re-implement against whichever backend.
- `THRESHOLD` above is a starting guess — tune it against real enrollment +
  test footage before trusting it on video frames.
- Next module: run `embedder.detect_and_embed()` per sampled video frame
  (not just single test images) and feed the per-frame matches into the
  event pipeline, alongside the weapon detector output.